In [8]:
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds
import numpy as np
import glob
import math
import json

# Stop numpy from truncating output 
np.set_printoptions(suppress=True)

In [9]:
ref_file = 'jan.TIFF'

with rasterio.open(ref_file) as src:
    ref_transform = src.transform
    ref_crs = src.crs
    ref_shape = (src.height, src.width)

print(f"Grid Shape: {ref_shape}")

Grid Shape: (1800, 3600)


In [10]:
dem_array = np.zeros(ref_shape, dtype=np.float32)
gebco_files = glob.glob("gebco_2025_geotiff/*.tif")

print("Downsampling and stitching GEBCO tiles...")
for file in gebco_files:
    with rasterio.open(file) as src:
        # Calculate exactly where this tile fits in the master array
        window = from_bounds(*src.bounds, transform=ref_transform)
        
        row_start, col_start = math.floor(window.row_off), math.floor(window.col_off)
        height, width = math.ceil(window.height), math.ceil(window.width)
        
        # Downsample from the disk
        data = src.read(
            1, 
            out_shape=(height, width), 
            resampling=Resampling.average
        )
        
        # Paste into the master elevation array
        dem_array[row_start:row_start+height, col_start:col_start+width] = data

print("GEBCO DEM processed successfully.")

Downsampling and stitching GEBCO tiles...
GEBCO DEM processed successfully.


In [11]:
lc_array = np.zeros(ref_shape, dtype=np.uint8)
print("Resampling Land Cover HDF to reference grid...")

with rasterio.open('MCD12C1.hdf') as hdf_src:
    # Subdataset 0 standard IGBP classification
    subdataset_name = hdf_src.subdatasets[0] 
    
    with rasterio.open(subdataset_name) as lc_src:
        reproject(
            source=rasterio.band(lc_src, 1),
            destination=lc_array,
            src_transform=lc_src.transform,
            src_crs=lc_src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest # Nearest neighbor
        )

print("[DONE] Land Cover aligned.")

Resampling Land Cover HDF to reference grid...


c:\Users\snes_\miniconda3\envs\geo_env\Lib\site-packages\rasterio\__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


[DONE] Land Cover aligned.


In [12]:
# Get row indices
rows = np.arange(ref_shape[0])
cols = np.zeros(ref_shape[0])

# Transform pixel coordinates to geographic latitudes
_, lats = rasterio.transform.xy(ref_transform, rows, cols)

# Calculate weights based on the cosine of the latitude
lats_rad = np.radians(lats)
weights_1d = np.cos(lats_rad)

# 1D column weights across the entire 2D array
weight_array = np.broadcast_to(weights_1d[:, np.newaxis], ref_shape)
print("[DONE] Latitude weights calculated.")

[DONE] Latitude weights calculated.


The cosine function corrects for spatial area distortion inherent in standard geographic map grids.

In an unprojected geographic coordinate system, the data grid assumes every pixel represents the same angular degree spacing (e.g., 0.1° latitude by 0.1° longitude). The physical area of these pixels is not constant in the real world. Longitude lines converge at the poles, meaning a pixel at the equator is physically much wider than a pixel near the Arctic.

Assuming the Earth is a sphere with radius $R$, the circumference of the equator is $2\pi R$. The circumference of any other latitude band $\theta$ shrinks as you move away from the equator.

The radius of the latitude circle $r$ at latitude $\theta$ is:


$$r = R \cos(\theta)$$

Because the physical width of a longitude degree depends strictly on this radius, the physical area $A$ of a pixel at latitude $\theta$ is directly proportional to the cosine of that latitude:


$$A(\theta) = A_{equator} \cos(\theta)$$


The script calculates the mean color of different land types by averaging pixel RGB values. If I use standard mean operation without weights, it severely overrepresents polar regions. A small block of ice in Antarctica would get the exact same voting power as a massive block of forest in Brazil simply because they both occupy exactly one pixel in the TIFF array.

The `weights_1d` array fixes this technical flaw by scaling the voting power of each pixel row based on its physical size:

* **Equator (0°):** $\cos(0) = 1$ (100% weight, largest physical area)
* **New York (40°):** $\cos(40^\circ \cdot \frac{\pi}{180}) \approx 0.766$ (76.6% weight)
* **Oslo (60°):** $\cos(60^\circ \cdot \frac{\pi}{180}) = 0.5$ (50% weight)
* **Poles (90°):** $\cos(90^\circ \cdot \frac{\pi}{180}) = 0$ (0% weight)

Passing `weights=cat_weights` into `np.average()`, numpy multiplies the RGB value of each pixel by its corresponding cosine fraction before taking the average. Just to make sure the final hex colors accurately represent the true physical area of the Earth's surface.

In [13]:
# Calculate topographic roughness (gradient magnitude)
dy, dx = np.gradient(dem_array)
roughness = np.sqrt(dx**2 + dy**2)

masks = {
    "Oceans": (lc_array == 0) & (dem_array < 0),
    "Fresh Water": (lc_array == 0) & (dem_array >= 0),
    "Snow and Ice": (lc_array == 15),
    "Deserts": (lc_array == 16),
    "Forests": np.isin(lc_array, [1, 2, 3, 4, 5]),
    "Grasslands": np.isin(lc_array, [8, 9, 10, 14]), # Includes savannas/mosaics
    "Shrublands": np.isin(lc_array, [6, 7]),
    "Croplands": (lc_array == 12),
    "Wetlands": (lc_array == 11),
    "Urban Areas": (lc_array == 13),
    "Mountains": (dem_array > 1000) & (roughness > 50), # Elevated and rugged
    "Total Land Mean": (dem_array >= 0) & (lc_array != 0), 
    "Total Ocean Mean": (dem_array < 0),
    "Total Earth Mean": np.ones(ref_shape, dtype=bool) 
}
print("[DONE] Masks defined.")

[DONE] Masks defined.


In [14]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sept', 'oct', 'nov', 'dec']
final_data = []

def rgb_to_hex(r, g, b):
    return f"#{max(0, min(255, int(round(r)))):02x}{max(0, min(255, int(round(g)))):02x}{max(0, min(255, int(round(b)))):02x}"

for month in months:
    filename = f"{month}.TIFF"
    print(f"Extracting colors for {month}...")
    
    with rasterio.open(filename) as src:
        r = src.read(1).astype(float)
        g = src.read(2).astype(float)
        b = src.read(3).astype(float)
        
        # dataset_mask() returns 255 for valid pixels, 0 for nodata
        valid_data = src.dataset_mask() > 0
        
        month_results = {"month": month}
        
        for cat_name, mask in masks.items():
            combined_mask = mask & valid_data
            cat_weights = weight_array[combined_mask]
            
            if np.sum(cat_weights) == 0:
                print(f"  WARNING: No valid pixels for '{cat_name}' in {month} — check your masks.")
                month_results[cat_name] = "#000000"
                continue
            
            mean_r = np.average(r[combined_mask], weights=cat_weights)
            mean_g = np.average(g[combined_mask], weights=cat_weights)
            mean_b = np.average(b[combined_mask], weights=cat_weights)
            
            month_results[cat_name] = rgb_to_hex(mean_r, mean_g, mean_b)
        
    final_data.append(month_results)

print("[DONE] All months processed!")

output_file = 'earth_hues.json'
with open(output_file, 'w') as f:
    json.dump(final_data, f, indent=4)

print(f"[DONE] Extraction complete and saved to {output_file}")

Extracting colors for jan...
Extracting colors for feb...
Extracting colors for mar...
Extracting colors for apr...
Extracting colors for may...
Extracting colors for jun...
Extracting colors for jul...
Extracting colors for aug...
Extracting colors for sept...
Extracting colors for oct...
Extracting colors for nov...
Extracting colors for dec...
[DONE] All months processed!
[DONE] Extraction complete and saved to earth_hues.json


In [15]:
output_file = 'earth_hues.json'

with open(output_file, 'w') as f:
    json.dump(final_data, f, indent=4)
    
print(f"Extraction complete! Data saved to {output_file}")

Extraction complete! Data saved to earth_hues.json
